### Setup and Imports

In [1]:
# ClaimVerify — Hybrid Offline Retrieval + Google API Fallback (Deliverable 3)

import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

# -------------------------------
# Paths / Config
# -------------------------------
BASE_PATH = Path("/Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject")

DATA_PATH = BASE_PATH / "data/processed/merged_factcheck_datasetcleaned.csv"

# Existing embeddings / index from Deliverable 2
EMBED_DIR = BASE_PATH / "data/processed/embeddings"
FAISS_DIR = BASE_PATH / "data/processed/faiss_index"

EMBED_FILE = EMBED_DIR / "claim_embeddings.npy"
META_EMBED_FILE = EMBED_DIR / "claim_metadata.csv"

FAISS_INDEX_FILE = FAISS_DIR / "claimverify_faiss_index.bin"
FAISS_META_FILE = FAISS_DIR / "claimverify_faiss_metadata.csv"

# SentenceTransformer model name
ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Threshold for hybrid fallback
SIMILARITY_THRESHOLD = 0.70   # Deliverable 3 improvement: explicit low-sim threshold

# Random seed for any sampling
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Environment & paths initialized.")
print(f"Base path: {BASE_PATH}")

Environment & paths initialized.
Base path: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject


### Text Normalization (Aligned with this Classifier)

In [2]:
import re
import string

def normalize_claim_text(text: str) -> str:
    """
    Deliverable 3 improvement:
    - Same normalization used in classifier training (lowercase, de-noising, whitespace fixes).
    - Applied at retrieval time so embeddings see cleaner, consistent text.
    """
    if not isinstance(text, str):
        text = str(text)

    # Lowercase
    text = text.lower()

    # Replace fancy quotes / dashes
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = text.replace("–", "-").replace("—", "-")

    # Remove leftover HTML-like tags if any
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove extra punctuation while keeping key sentence markers
    # Keep . , ! ? - % $ but collapse repeated ones
    allowed = set(".!,?-%$")
    cleaned_chars = []
    for ch in text:
        if ch.isalnum() or ch.isspace() or ch in allowed or ch in ['"', "'"]:
            cleaned_chars.append(ch)
        # everything else is dropped
    text = "".join(cleaned_chars)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


### Load Dataset and Inspect Cleaned Text

In [3]:
# Load processed dataset
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with shape: {df.shape}")

# Keep only relevant columns
required_cols = ["claim_id", "claim_text", "verdict_mapped", "summary", "url", "dataset_source"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in dataset: {missing}")

# Apply normalization (Deliverable 3 improvement)
df["claim_text_clean"] = df["claim_text"].astype(str).apply(normalize_claim_text)

print("\nSample cleaned rows:")
display(df[["claim_id", "claim_text", "claim_text_clean", "verdict_mapped"]].head(10))

print("\nLabel distribution:")
print(df["verdict_mapped"].value_counts(normalize=True))

Loaded dataset with shape: (25540, 9)

Sample cleaned rows:


,claim_id,claim_text,claim_text_clean,verdict_mapped
0,P00001,John McCain opposed bankruptcy protections for...,john mccain opposed bankruptcy protections for...,Likely True
1,P00002,"""Bennie Thompson actively cheer-led riots in t...","""bennie thompson actively cheer-led riots in t...",Likely False
2,P00003,"Says Maggie Hassan was ""out of state on 30 day...","says maggie hassan was ""out of state on 30 day...",Likely True
3,P00004,"""BUSTED: CDC Inflated COVID Numbers, Accused o...","""busted cdc inflated covid numbers, accused of...",Likely False
4,P00005,"""I'm the only (Republican) candidate that has ...","""i'm the only republican candidate that has ac...",Uncertain
5,P00006,"""There are actually only 30 countries that pra...","""there are actually only 30 countries that pra...",Likely True
6,P00007,"""My husband and I have never gotten a penny of...","""my husband and i have never gotten a penny of...",Likely False
7,P00008,"""If you go strictly by the numbers, crime is d...","""if you go strictly by the numbers, crime is d...",Likely True
8,P00009,"""The American people say, don't touch Social S...","""the american people say, don't touch social s...",Likely True
9,P00010,"""Since 1978, CEO compensation rose over 1,000%...","""since 1978, ceo compensation rose over 1,000%...",Likely True



Label distribution:
verdict_mapped
Likely False    0.563430
Likely True     0.255521
Uncertain       0.181049
Name: proportion, dtype: float64


### Load FAISS Index & Metadata

In [5]:
# Load FAISS index and metadata (Deliverable 3 still uses same binaries, but with improved query preprocessing)
print("📂 Loading FAISS index and metadata...")

if not FAISS_INDEX_FILE.exists():
    raise FileNotFoundError(f"FAISS index file not found at {FAISS_INDEX_FILE}")
if not FAISS_META_FILE.exists():
    raise FileNotFoundError(f"FAISS metadata file not found at {FAISS_META_FILE}")

index = faiss.read_index(str(FAISS_INDEX_FILE))
metadata = pd.read_csv(FAISS_META_FILE)

print(f"✅ Loaded FAISS index with {index.ntotal} vectors.")
print(f" Metadata shape: {metadata.shape}")
display(metadata.head(5))

📂 Loading FAISS index and metadata...
✅ Loaded FAISS index with 25540 vectors.
 Metadata shape: (25540, 6)


,claim_id,claim_text,verdict_mapped,summary,url,dataset_source
0,P00001,John McCain opposed bankruptcy protections for...,Likely True,NaN,https://www.politifact.com/factchecks/2008/jun...,PolitiFact
1,P00002,"""Bennie Thompson actively cheer-led riots in t...",Likely False,NaN,https://www.politifact.com/factchecks/2022/jun...,PolitiFact
2,P00003,"Says Maggie Hassan was ""out of state on 30 day...",Likely True,NaN,https://www.politifact.com/factchecks/2016/may...,PolitiFact
3,P00004,"""BUSTED: CDC Inflated COVID Numbers, Accused o...",Likely False,NaN,https://www.politifact.com/factchecks/2021/feb...,PolitiFact
4,P00005,"""I'm the only (Republican) candidate that has ...",Uncertain,NaN,https://www.politifact.com/factchecks/2015/aug...,PolitiFact


### Load SentenceTransformer Model for Retrieval

In [6]:
print("⚙️ Loading SentenceTransformer retrieval model...")
retrieval_model = SentenceTransformer(ST_MODEL_NAME)
print("✅ SentenceTransformer loaded.")

⚙️ Loading SentenceTransformer retrieval model...
✅ SentenceTransformer loaded.


### Google API Fallback Stub (Deliverable 3 Improvement)

In [7]:
def google_factcheck_fallback_stub(query: str, top_k: int = 3):
    """
    Deliverable 3 improvement:
    - Stub for Google Custom Search / Fact Check API.
    - No real API calls; returns structured mock evidence.
    - Used when FAISS similarity is below a threshold.
    """
    cleaned_query = normalize_claim_text(query)

    # In a real system, you'd call Google APIs here.
    # For Deliverable 3, we simulate plausible fact-check hits.
    mock_results = []

    for i in range(top_k):
        mock_results.append({
            "rank": i + 1,
            "claim_text": f"Mock fact-check result {i+1} for: {cleaned_query}",
            "similarity": round(0.40 + 0.05 * (top_k - i), 3),  # arbitrary descending scores
            "verdict_mapped": random.choice(["Likely True", "Likely False", "Uncertain"]),
            "summary": "This is a placeholder snippet representing external fact-check content.",
            "url": f"https://example-fact-check.org/article/{i+1}",
            "dataset_source": "GoogleHybridStub"
        })

    return mock_results

### Hybrid Retrieval Function with Latency Logging

In [8]:
def hybrid_retrieve_claims(
    user_claim: str,
    top_k: int = 5,
    similarity_threshold: float = SIMILARITY_THRESHOLD,
):
    """
    Hybrid retrieval pipeline for Deliverable 3.

    1. Normalize user claim (same as classifier preprocessing).
    2. Embed with MiniLM and search FAISS index.
    3. If top-1 similarity < threshold → call Google fallback stub.
    4. Log retrieval latency.
    5. Return structured result dict.
    """

    # Step 0: basic safety
    if not isinstance(user_claim, str) or not user_claim.strip():
        raise ValueError("Please provide a non-empty claim string.")

    claim_clean = normalize_claim_text(user_claim)

    # Step 1: Offline retrieval with latency logging
    t0 = time.perf_counter()
    query_vec = retrieval_model.encode([claim_clean], normalize_embeddings=True)
    offline_encode_time = (time.perf_counter() - t0) * 1000  # ms

    t1 = time.perf_counter()
    scores, indices = index.search(query_vec, top_k)
    offline_search_time = (time.perf_counter() - t1) * 1000  # ms

    offline_latency_ms = offline_encode_time + offline_search_time

    # Build offline result table
    retrieved = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        row = metadata.iloc[idx]
        retrieved.append({
            "rank": rank + 1,
            "claim_id": row.get("claim_id", None),
            "claim_text": row.get("claim_text", ""),
            "similarity": float(score),
            "verdict_mapped": row.get("verdict_mapped", None),
            "summary": row.get("summary", None),
            "url": row.get("url", None),
            "dataset_source": row.get("dataset_source", None)
        })

    top1_similarity = float(scores[0][0]) if len(scores[0]) > 0 else 0.0

    # Step 2: Decide source: OfflineDB vs Hybrid Fallback
    if top1_similarity >= similarity_threshold:
        # Pure offline case
        total_latency_ms = offline_latency_ms
        source = "OfflineDB"
        evidence = retrieved
    else:
        # Hybrid Fallback
        t2 = time.perf_counter()
        google_results = google_factcheck_fallback_stub(user_claim, top_k=top_k)
        fallback_time_ms = (time.perf_counter() - t2) * 1000

        total_latency_ms = offline_latency_ms + fallback_time_ms
        source = "Hybrid Fallback"

        # Combine info: offline suggestions + stubbed external hits
        evidence = retrieved + google_results

    result = {
        "query": user_claim,
        "query_clean": claim_clean,
        "top1_similarity": top1_similarity,
        "similarity_threshold": similarity_threshold,
        "source": source,                    # Deliverable 3 improvement: explicit source flag
        "offline_latency_ms": round(offline_latency_ms, 2),
        "total_latency_ms": round(total_latency_ms, 2),
        "offline_results": retrieved,
        "evidence": evidence                 # what UI should actually display
    }

    return result


### Quick Manual Test of Hybrid Retrieval

In [9]:
test_claim = "COVID-19 vaccines cause infertility in women."

hybrid_result = hybrid_retrieve_claims(test_claim, top_k=5)

print("Query           :", hybrid_result["query"])
print("Normalized query:", hybrid_result["query_clean"])
print("Top1 similarity :", round(hybrid_result["top1_similarity"], 3))
print("Threshold       :", hybrid_result["similarity_threshold"])
print("Source used     :", hybrid_result["source"])
print("Offline latency :", hybrid_result["offline_latency_ms"], "ms")
print("Total latency   :", hybrid_result["total_latency_ms"], "ms")
print("\nTop evidence rows:")

evidence_df = pd.DataFrame(hybrid_result["evidence"])
display(evidence_df.head(10))

Query           : COVID-19 vaccines cause infertility in women.
Normalized query: covid-19 vaccines cause infertility in women.
Top1 similarity : 0.836
Threshold       : 0.7
Source used     : OfflineDB
Offline latency : 691.23 ms
Total latency   : 691.23 ms

Top evidence rows:


,rank,claim_id,claim_text,similarity,verdict_mapped,summary,url,dataset_source
0,1,P20071,Women’s menstrual cycles and fertility are aff...,0.835647,Likely False,NaN,https://www.politifact.com/factchecks/2021/apr...,PolitiFact
1,2,P12748,University of Miami researchers have found tha...,0.759346,Likely False,NaN,https://www.politifact.com/factchecks/2021/may...,PolitiFact
2,3,P09341,"A study found an ""82% miscarriage rate"" among ...",0.702567,Likely False,NaN,https://www.politifact.com/factchecks/2021/jul...,PolitiFact
3,4,P20583,"Officials ""recommend that women who get one of...",0.682969,Likely False,NaN,https://www.politifact.com/factchecks/2021/apr...,PolitiFact
4,5,P00936,The COVID-19 vaccines cause AIDS.,0.666309,Likely False,NaN,https://www.politifact.com/factchecks/2021/dec...,PolitiFact


### Recall@k Evaluation (Self-Retrieval)

In [10]:
def evaluate_recall_at_k(
    k_list = [1, 3, 5],
    n_samples: int = 1000,
):
    """
    Deliverable 3 improvement:
    - Quantitative retrieval evaluation using self-retrieval.
    - For each sampled claim, we try to retrieve its own entry by text.
    - Compute Recall@k over all sampled queries.

    NOTE: only uses offline FAISS retrieval (hybrid fallback not needed here).
    """
    # Ensure we have at least n_samples rows
    total_rows = len(metadata)
    n_samples = min(n_samples, total_rows)

    # Sample indices
    sampled_indices = np.random.choice(total_rows, size=n_samples, replace=False)

    # Mapping from claim_id to index position in metadata, if present
    if "claim_id" in metadata.columns:
        id_to_pos = {cid: idx for idx, cid in enumerate(metadata["claim_id"])}
    else:
        id_to_pos = None

    recalls = {k: 0 for k in k_list}

    print(f"Evaluating Recall@k on {n_samples} sampled claims...")
    t_start = time.perf_counter()

    for i, idx in enumerate(sampled_indices, start=1):
        row = metadata.iloc[idx]
        claim_id = row.get("claim_id", None)
        text = row["claim_text"]

        # Normalize like queries
        text_clean = normalize_claim_text(text)
        query_vec = retrieval_model.encode([text_clean], normalize_embeddings=True)
        scores, inds = index.search(query_vec, max(k_list))

        # Convert retrieved indices to claim_ids if possible
        retrieved_ids = [
            metadata.iloc[j].get("claim_id", None) for j in inds[0]
        ]

        for k in k_list:
            if claim_id is not None and claim_id in retrieved_ids[:k]:
                recalls[k] += 1

        if i % 200 == 0:
            print(f"Processed {i}/{n_samples} queries...")

    t_end = time.perf_counter()
    total_time = t_end - t_start

    recall_at_k = {k: recalls[k] / n_samples for k in k_list}

    print("\n📊 Recall@k Results (offline self-retrieval)")
    for k in k_list:
        print(f"Recall@{k}: {recall_at_k[k]:.4f}")

    print(f"\nTotal evaluation time: {total_time:.2f} seconds "
          f"({n_samples / total_time:.2f} queries/sec)")

    return recall_at_k

recall_metrics = evaluate_recall_at_k(k_list=[1,3,5], n_samples=800)  # you can adjust n_samples


Evaluating Recall@k on 800 sampled claims...
Processed 200/800 queries...
Processed 400/800 queries...
Processed 600/800 queries...
Processed 800/800 queries...

📊 Recall@k Results (offline self-retrieval)
Recall@1: 0.9962
Recall@3: 0.9988
Recall@5: 1.0000

Total evaluation time: 10.78 seconds (74.25 queries/sec)


### Cross-Domain Recall (PolitiFact vs Snopes)

In [11]:
def evaluate_recall_by_domain(
    k_list = [1, 3, 5],
    n_samples_per_domain: int = 400,
):
    """
    Deliverable 3 extension:
    - Evaluate retrieval robustness across domains.
    - PolitiFact vs Snopes (simulates cross-domain drift).
    """

    if "dataset_source" not in metadata.columns:
        raise ValueError("metadata must contain 'dataset_source' column for domain evaluation.")

    domains = metadata["dataset_source"].unique()
    results = {}

    for domain in domains:
        domain_df = metadata[metadata["dataset_source"] == domain].reset_index(drop=True)
        domain_size = len(domain_df)
        if domain_size == 0:
            continue

        ns = min(n_samples_per_domain, domain_size)
        sampled_indices = np.random.choice(domain_size, size=ns, replace=False)

        recalls = {k: 0 for k in k_list}
        print(f"\nEvaluating domain '{domain}' on {ns} samples...")

        t_start = time.perf_counter()
        for i, idx in enumerate(sampled_indices, start=1):
            row = domain_df.iloc[idx]
            claim_id = row.get("claim_id", None)
            text = row["claim_text"]

            text_clean = normalize_claim_text(text)
            query_vec = retrieval_model.encode([text_clean], normalize_embeddings=True)
            scores, inds = index.search(query_vec, max(k_list))

            retrieved_ids = [
                metadata.iloc[j].get("claim_id", None) for j in inds[0]
            ]

            for k in k_list:
                if claim_id is not None and claim_id in retrieved_ids[:k]:
                    recalls[k] += 1

        t_end = time.perf_counter()
        total_time = t_end - t_start

        recall_at_k = {k: recalls[k] / ns for k in k_list}
        results[domain] = recall_at_k

        print(f"Domain '{domain}' Recall@k:")
        for k in k_list:
            print(f"  Recall@{k}: {recall_at_k[k]:.4f}")
        print(f"  Eval time: {total_time:.2f}s ({ns / total_time:.2f} queries/sec)")

    return results

domain_recall = evaluate_recall_by_domain(k_list=[1,3,5], n_samples_per_domain=300)
domain_recall


Evaluating domain 'PolitiFact' on 300 samples...
Domain 'PolitiFact' Recall@k:
  Recall@1: 1.0000
  Recall@3: 1.0000
  Recall@5: 1.0000
  Eval time: 3.74s (80.11 queries/sec)

Evaluating domain 'Snopes' on 300 samples...
Domain 'Snopes' Recall@k:
  Recall@1: 0.9933
  Recall@3: 1.0000
  Recall@5: 1.0000
  Eval time: 3.70s (81.13 queries/sec)


{'PolitiFact': {1: 1.0, 3: 1.0, 5: 1.0},
 'Snopes': {1: 0.9933333333333333, 3: 1.0, 5: 1.0}}